In [1]:
import torch
import torch.nn as nn
import torch.nn.functional as F
import numpy as np

class SimplifiedNTM(nn.Module):
    def __init__(self, input_size, memory_size, memory_vector_dim):
        super(SimplifiedNTM, self).__init__()
        self.N, self.M = memory_size, memory_vector_dim
        
        self.register_buffer('memory', torch.zeros(self.N, self.M))
        
        self.controller = nn.Sequential(
            nn.Linear(input_size, 64),
            nn.ReLU(),
            nn.Linear(64, self.N + self.N + self.M + self.M)
        )

    def step(self, x):
        out = self.controller(x)
        
        read_logits = out[:, :self.N]
        write_logits = out[:, self.N:2*self.N]
        write_data = torch.tanh(out[:, 2*self.N:2*self.N + self.M])
        erase_mask = torch.sigmoid(out[:, 2*self.N + self.M:])
        
        read_weights = F.softmax(read_logits, dim=-1)
        write_weights = F.softmax(write_logits, dim=-1)
        
        read_vec = torch.matmul(read_weights, self.memory)
        
        erase_gate = torch.matmul(write_weights.unsqueeze(-1), erase_mask.unsqueeze(1))
        add_gate = torch.matmul(write_weights.unsqueeze(-1), write_data.unsqueeze(1))
        
        self.memory = self.memory * (1 - erase_gate.squeeze(0)) + add_gate.squeeze(0)
        
        return read_vec

    def reset_memory(self):
        self.memory.fill_(0)

N_MEM, M_DIM = 5, 4
input_dim = 8
ntm = SimplifiedNTM(input_dim, N_MEM, M_DIM)

data_sequence = [torch.randn(1, input_dim) for _ in range(3)]

print("Начало работы uNTM")
for i, x in enumerate(data_sequence):
    read_res = ntm.step(x)
    print(f"Шаг {i+1}:")
    print(f"  Прочитано из памяти: {read_res.detach().numpy().round(3)}")
    print(f"  Состояние памяти (фрагмент):\n{ntm.memory.detach().numpy().round(2)}")
    print("\n")

Начало работы uNTM
Шаг 1:
  Прочитано из памяти: [[0. 0. 0. 0.]]
  Состояние памяти (фрагмент):
[[ 0.05  0.02 -0.03  0.07]
 [ 0.06  0.02 -0.04  0.07]
 [ 0.04  0.02 -0.03  0.05]
 [ 0.03  0.01 -0.02  0.04]
 [ 0.07  0.03 -0.04  0.09]]


Шаг 2:
  Прочитано из памяти: [[ 0.052  0.019 -0.032  0.065]]
  Состояние памяти (фрагмент):
[[ 0.02  0.03 -0.09  0.09]
 [ 0.03  0.03 -0.08  0.09]
 [ 0.01  0.03 -0.08  0.08]
 [-0.01  0.03 -0.1   0.07]
 [ 0.03  0.04 -0.1   0.11]]


Шаг 3:
  Прочитано из памяти: [[ 0.018  0.03  -0.09   0.088]]
  Состояние памяти (фрагмент):
[[ 0.    0.21 -0.09  0.18]
 [ 0.02  0.16 -0.08  0.15]
 [-0.    0.14 -0.09  0.13]
 [-0.02  0.12 -0.1   0.12]
 [ 0.02  0.17 -0.11  0.17]]


